# Plans, recettes et production en série

**Cartomize · Version 1.0**

Durée indicative : 40 minutes.

## Objectifs

Construire une chaîne dépendante, exécuter des opérateurs enregistrés et décliner une carte en plusieurs livrables.

## Apport de Cartomize

Les références `@étape` relient automatiquement les sorties aux entrées suivantes. Les recettes conservent les choix cartographiques et les lots appliquent des variables.

Données : paysage pédagogique généré avec une graine fixe. Les cartes et mesures ne décrivent aucun site réel.

## Préparation

Installer l’environnement décrit dans le [guide de démarrage](../README.md), puis exécuter les cellules dans l’ordre. Chaque exécution utilise un nouveau dossier de résultats.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt
from IPython.display import display, Image
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'course.py').is_file() or (p/'tutorials/course.py').is_file())
root = root if (root/'course.py').is_file() else root/'tutorials'
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
import cartomize as cm
from course import demo, run_directory, show_raster, show_rgb, gallery, CRS, CLASSES, ROLES

In [2]:
out = run_directory("08_automatisation")
d = demo(out/"data")

## 1. Construire un plan de traitement

Le graphe est validé avant l’écriture. Chaque opération publie un produit principal et, selon le cas, des produits secondaires. Une référence évite de recopier le chemin d’une sortie intermédiaire.

In [3]:
steps=[
 {'id':'ndvi','operation':'indices','parameters':{'source':str(d['image']),'indices':['NDVI']}},
 {'id':'focal','operation':'focal','parameters':{'source':'@ndvi','statistic':'mean','size':5}},
 {'id':'zones','operation':'raster.zonal_stats','parameters':{'source':'@focal','zones':str(d['zones'])}},
]
plan=cm.processing_plan(steps)
cm.validate_plan(plan)
result=cm.run_plan(plan,out/'chaine',workers=2,block_size=64)
display(result)

PosixPath('tutorials/outputs/08_automatisation_i5ifykac/chaine/automation.json')

## 2. Ajouter une carte finale

Le plan cartographique analyse les entrées, propose une mise en page et exécute les dépendances. Les règles restent explicites ; le plan doit être examiné avant une production destinée à publication.

In [4]:
assessment=cm.assess_project([d['before'],d['points']],goal='landcover')
proposals=cm.propose_layouts('landcover',classes=3,count=3)
display(pd.DataFrame(proposals)[['id','score']])
cartographic=cm.plan_cartography([d['before'],d['points']],goal='landcover',
    title='Couverture simulée',credits='Données synthétiques',template='occupation_sol/institutionnel')
cm.run_plan(cartographic,out/'plan_cartographique',dpi=120,formats=('png',))

Out[4]: PosixPath('tutorials/outputs/08_automatisation_i5ifykac/plan_cartographique/automation.json')


,id,score
0,occupation_sol/institutionnel,131.52
1,professionnelles/28-carte-panoramique-deux-volets,124.35
2,topographique/institutionnel,94.68


## 3. Enregistrer et instancier une recette

In [5]:
carte=cm.Map(title='${titre}',crs=CRS,credits='Données synthétiques')
carte.add_layer(d['before'],name='Couverture',classes=CLASSES)
recipe=cm.save_recipe(carte,out/'recette.json')
loaded=cm.load_recipe(recipe)
instance=cm.instantiate_recipe(loaded,variables={'titre':'État initial'})
instance.export(out/'recette_initiale.pdf',dpi=150)
cm.run_recipe(recipe,out/'recette_executee',variables={'titre':'État de référence'},formats=('png',),dpi=120)
manifest={'schema':'cartomize.batch.v1','recipe':str(recipe),'dpi':120,'jobs':[
 {'job_id':'t0','output_name':'etat_initial','variables':{'titre':'État initial'},'output_formats':['png']},
 {'job_id':'t1','output_name':'etat_final','variables':{'titre':'État final'},'layer_bindings':{'Couverture':str(d['after'])},'output_formats':['png']},
]}
ledger=cm.run_batch(manifest,out/'lot')
status=json.loads(ledger.read_text())
assert status['complete']
display(pd.DataFrame(status['jobs']))

,job_id,status,outputs
0,t0,completed,[/workspace/scratch/993477e68500/cartomize-wor...
1,t1,completed,[/workspace/scratch/993477e68500/cartomize-wor...


## 4. Utiliser le registre

Les 34 opérateurs sont décrits avec leurs paramètres. Ils servent à l’interface, à l’API et aux plans. La [matrice de couverture](../COVERAGE.md) indique le cours de chaque famille.

In [6]:
registry=pd.DataFrame(cm.operation_catalog())
assert len(registry)==34
display(registry[['id','label','output']])
operation=cm.execute_operation('vector.buffer',{'data':str(d['points']),'distance':300,'metric_crs':CRS},out/'operation')
assert Path(operation['primary']).is_file()

,id,label,output
0,vector.clip,Découpage vectoriel,vector
1,vector.buffer,Zone tampon,vector
2,vector.sjoin,Jointure spatiale,vector
3,vector.nearest,Jointure de proximité,vector
4,vector.dissolve,Dissolution,vector
5,vector.reproject,Reprojection vectorielle,vector
6,vector.make_valid,Réparation des géométries,vector
7,vector.intersection,Intersection,vector
8,vector.union,Union,vector
9,vector.difference,Différence,vector


## Exercice

Ajouter au plan une étape de reclassification après le filtre focal. Pourquoi une opération qui utilise `@inconnu` doit-elle être refusée avant l’exécution ?

<details>
<summary>Éléments de correction</summary>

La référence doit désigner une étape antérieure avec un produit compatible. La validation du graphe empêche une chaîne incomplète. Pour un seuil continu, utiliser l’opérateur `calculate` et une expression `where`.

</details>

## Reproduction

Les paramètres, la graine et les chemins sont explicites dans les cellules. Le répertoire `out` contient les produits de cette exécution. Adapter les sources, le système de coordonnées et les paramètres avant de transférer la méthode à une étude.